# Feature Engineering Test

Notebook nay kiem tra viec tao feature cho bai toan spam/not spam. Phien ban cu doc `../data/raw/combined_data.csv`, nhung file do khong co trong repo hien tai, nen notebook chuyen sang raw SpamAssassin co san.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from config import RAW_DATA_DIR, SPAMASSASSIN_ARCHIVES, FIGURES_DIR, REPORTS_DIR
from src.data_loader import parse_email_file, iter_email_paths
from src.feature_engineering import build_tfidf_features, extract_manual_features, combine_features

FAST_MODE_ROWS_PER_CLASS = 500
TEXT_SAMPLE_CACHE = REPORTS_DIR / "spamassassin_text_sample.csv"
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

if TEXT_SAMPLE_CACHE.exists():
    df = pd.read_csv(TEXT_SAMPLE_CACHE)
else:
    rows = []
    for source in SPAMASSASSIN_ARCHIVES:
        extract_dir = RAW_DATA_DIR / Path(source["file_name"]).stem.replace(".tar", "")
        if not extract_dir.exists():
            continue
        for email_path in iter_email_paths(extract_dir):
            try:
                subject, text = parse_email_file(email_path)
            except Exception:
                continue
            if len(text.strip()) >= 20:
                rows.append({"source": source["name"], "subject": subject, "label": int(source["label"]), "text": text})
    df = pd.DataFrame(rows).drop_duplicates(subset=["label", "text"]).reset_index(drop=True)
    sample_parts = []
    for _, part in df.groupby("label"):
        sample_parts.append(part.sample(n=min(len(part), FAST_MODE_ROWS_PER_CLASS), random_state=42))
    df = pd.concat(sample_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)
    df.to_csv(TEXT_SAMPLE_CACHE, index=False, encoding="utf-8")

sample_parts = []
for _, part in df.groupby("label"):
    sample_parts.append(part.sample(n=min(len(part), FAST_MODE_ROWS_PER_CLASS), random_state=42))
df = pd.concat(sample_parts, ignore_index=True).sample(frac=1, random_state=42).reset_index(drop=True)

print("Rows used for feature test:", len(df))
display(df["label"].value_counts().sort_index().rename(index={0: "not spam", 1: "spam"}))
display(df.head())

## 1. Tao TF-IDF va manual features

TF-IDF bieu dien noi dung email, con manual features bam theo de bai: do dai, URL, dau `!`, ky tu `$`, ty le chu hoa.

In [ ]:
texts = df["text"].head(5).fillna("").astype(str).tolist()
manual_preview = extract_manual_features(texts)
display(manual_preview)

tfidf_preview, preview_vectorizer = build_tfidf_features(texts, max_features=20, ngram_range=(1, 2))
print("TF-IDF preview shape:", tfidf_preview.shape)
print("Sample tokens:", preview_vectorizer.get_feature_names_out()[:20])

## 2. Test model nho voi TF-IDF + manual features

Cell duoi train nhanh Logistic Regression de kiem tra feature co dung duoc trong pipeline phan loai hay khong. Day la test feature, khong thay the model train chinh trong `05_training_and_evaluation.ipynb`.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

texts = df["text"].fillna("").astype(str).tolist()
labels = df["label"].astype(int).tolist()

tfidf_matrix, vectorizer = build_tfidf_features(texts, max_features=2000, ngram_range=(1, 2))
manual_df = extract_manual_features(texts)
combined_features = combine_features(tfidf_matrix, manual_df)

X_train, X_test, y_train, y_test = train_test_split(
    combined_features, labels, test_size=0.2, random_state=42, stratify=labels
)

model = LogisticRegression(max_iter=500, random_state=42)
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

metrics = {
    "accuracy": accuracy_score(y_test, y_pred),
    "precision_spam": precision_score(y_test, y_pred, zero_division=0),
    "recall_spam": recall_score(y_test, y_pred, zero_division=0),
    "f1_spam": f1_score(y_test, y_pred, zero_division=0),
}
display(pd.DataFrame([metrics]))
print("TF-IDF feature count:", len(vectorizer.get_feature_names_out()))
print("Manual features:", manual_df.columns.tolist())

## 3. Correlation heatmap cua manual features

Heatmap nay duoc luu trong `reports/figures/` de notebook tong va bao cao dung lai.

In [ ]:
from IPython.display import Image, display

heatmap_path = FIGURES_DIR / "manual_feature_correlation_heatmap.png"
if heatmap_path.exists():
    display(Image(filename=str(heatmap_path)))
else:
    print("Heatmap not found. Run 02_eda.ipynb first.")